In [1]:
# Name  : Muhammad Salman Bin Hasnul Hizam
# ID    : IS01083889
# Course: CISB5123 Text Analytics
# Lab Assignment 3 – Topic Modeling using LDA (Gensim)

In [2]:

!pip install gensim nltk pyLDAvis --quiet

In [3]:
!pip install gensim nltk pyLDAvis --quiet

In [4]:
import pandas as pd
import re
import nltk
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/2ae3a15c-0ba3-4f9a-887f-
[nltk_data]     4bf317483540/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/2ae3a15c-0ba3-4f9a-887f-
[nltk_data]     4bf317483540/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/2ae3a15c-0ba3-4f9a-887f-
[nltk_data]     4bf317483540/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/2ae3a15c-0ba3-4f9a-887f-
[nltk_data]     4bf317483540/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
df = pd.read_csv('news_dataset.csv')

# Use only the 'text' column and remove null values
df = df[['text']].dropna()

print(f"Total documents: {len(df)}")
df.head()

Total documents: 11096


,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer    = PorterStemmer()

def preprocess(text):
    # Lowercase and remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    
    # Tokenize
    tokens = text.split()
    
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    # Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    
    return tokens

df['processed'] = df['text'].apply(preprocess)

print("Sample processed tokens:")
print(df['processed'].iloc[0])

Sample processed tokens:
['wonder', 'anyon', 'could', 'enlighten', 'car', 'saw', 'day', 'door', 'sport', 'car', 'look', 'late', 'earli', 'call', 'bricklin', 'door', 'realli', 'small', 'addit', 'front', 'bumper', 'separ', 'rest', 'bodi', 'know', 'anyon', 'tellm', 'model', 'name', 'engin', 'spec', 'year', 'product', 'car', 'made', 'histori', 'whatev', 'info', 'funki', 'look', 'car', 'pleas', 'email']


In [7]:
# Create dictionary
dictionary = corpora.Dictionary(df['processed'])

# Filter extremes to reduce noise
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create Bag-of-Words corpus
corpus = [dictionary.doc2bow(doc) for doc in df['processed']]

print(f"Dictionary size : {len(dictionary)}")
print(f"Corpus size     : {len(corpus)}")

Dictionary size : 10544
Corpus size     : 11096


In [8]:
NUM_TOPICS = 4

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=10,
    alpha='auto',
    eta='auto'
)

print("LDA model trained successfully!")

LDA model trained successfully!


In [9]:
print("=" * 60)
print(f"LDA Topics (num_topics = {NUM_TOPICS})")
print("=" * 60)

for idx, topic in lda_model.print_topics(num_words=10):
    print(f"\nTopic {idx + 1}: {topic}")

LDA Topics (num_topics = 4)

Topic 1: 0.010*"peopl" + 0.009*"would" + 0.008*"one" + 0.006*"govern" + 0.005*"say" + 0.005*"god" + 0.005*"think" + 0.005*"law" + 0.004*"right" + 0.004*"state"

Topic 2: 0.007*"get" + 0.007*"one" + 0.007*"would" + 0.007*"year" + 0.007*"dont" + 0.006*"like" + 0.006*"think" + 0.006*"know" + 0.006*"go" + 0.005*"time"

Topic 3: 0.024*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.008*"anonym" + 0.008*"inform" + 0.007*"univers" + 0.007*"new" + 0.006*"internet" + 0.006*"list" + 0.005*"april" + 0.005*"space" + 0.005*"nation"

Topic 4: 0.018*"use" + 0.012*"key" + 0.008*"encrypt" + 0.007*"system" + 0.007*"file" + 0.007*"chip" + 0.007*"one" + 0.006*"get" + 0.006*"program" + 0.005*"bit"


In [10]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=df['processed'],
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print(f"\nCoherence Score (c_v): {coherence_score:.4f}")


Coherence Score (c_v): 0.5238


In [11]:
interpretation = """
===========================================================
Name  : Muhammad Salman Bin Hasnul Hizam
ID    : IS01083889

Interpretation of the Coherence Score
===========================================================
The LDA model was trained on the news_dataset with 4 topics
and evaluated using the c_v coherence metric, which measures
how semantically similar the top words within each topic are
to one another. The coherence score obtained indicates the
quality of the discovered topics. A c_v score closer to 1.0
reflects highly coherent, meaningful topics where top words
frequently co-occur in similar contexts, while a score closer
to 0.0 suggests that the words within topics are unrelated.
In this experiment, the coherence score suggests that the
LDA model has identified moderately to well-defined topics
from the news articles. The 4 topics appear to capture
distinct thematic clusters within the dataset, such as
technology, sports, politics, or general news discussions.
Overall, the result confirms that LDA with proper text
pre-processing (stopword removal, stemming, and lemmatization)
is an effective unsupervised method for topic discovery in
unlabelled news corpora.
===========================================================
"""
print(interpretation)


Name  : Muhammad Salman Bin Hasnul Hizam
ID    : IS01083889

Interpretation of the Coherence Score
The LDA model was trained on the news_dataset with 4 topics
and evaluated using the c_v coherence metric, which measures
how semantically similar the top words within each topic are
to one another. The coherence score obtained indicates the
quality of the discovered topics. A c_v score closer to 1.0
reflects highly coherent, meaningful topics where top words
frequently co-occur in similar contexts, while a score closer
to 0.0 suggests that the words within topics are unrelated.
In this experiment, the coherence score suggests that the
LDA model has identified moderately to well-defined topics
from the news articles. The 4 topics appear to capture
distinct thematic clusters within the dataset, such as
technology, sports, politics, or general news discussions.
Overall, the result confirms that LDA with proper text
pre-processing (stopword removal, stemming, and lemmatization)
is an effecti